In [8]:
import os, copy, math, warnings
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

assert torch.cuda.is_available(), "No GPU: Settings > Accelerator > GPU T4 x2, then restart"
DEV = "cuda"

BASE = Path("/kaggle/input/notebooks/prahlad214/rt-detr-pipeline")
OUT = Path("/kaggle/working")

SEED, EPOCHS, BS, PATIENCE, NUM_WORKERS = 42, 15, 32, 5, 4
RETENTION_TARGET = 0.95            # keep >= 95% of true detections; threshold is fixed on validation
np.random.seed(SEED)

# (torchvision builder, input size, backbone lr)
MODELS = {
    "DenseNet201":      ("densenet201",       224, 1e-4),
    "InceptionV3":      ("inception_v3",      299, 1e-4),
    "ResNet101":        ("resnet101",         224, 1e-4),
    "ConvNeXt-Small":   ("convnext_small",    224, 5e-5),
    "EfficientNetV2-S": ("efficientnet_v2_s", 288, 2e-4),
}

In [10]:
CSV  = BASE / "test_candidate_proposals.csv"
root = BASE / "candidate_rois" / "test"
df = pd.read_csv(CSV)

df["path"] = [str(root / os.path.basename(p)) for p in df.crop_path]
df = df[df.path.map(os.path.exists)].reset_index(drop=True)
y, groups = df.is_tp.astype(int).values, df.patientId.values

# patient-level split ~67/17/17, stratified by is_tp (no patient appears in two splits)
idx = np.arange(len(df))
rest, te = next(StratifiedGroupKFold(6, shuffle=True, random_state=SEED).split(idx, y, groups))
a, b = next(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(rest, y[rest], groups[rest]))
tr, va = rest[a], rest[b]
print(f"crops {len(df)} | patients {df.patientId.nunique()} | train/val/test {len(tr)}/{len(va)}/{len(te)}")
imgs = [Image.open(p).convert("RGB") for p in df.path]            # cached in RAM

NORM = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
def tfs(sz):
    train = T.Compose([T.Resize((sz, sz)), T.RandomHorizontalFlip(),
                       T.RandomAffine(10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
                       T.ColorJitter(0.2, 0.2), T.ToTensor(), NORM])
    return train, T.Compose([T.Resize((sz, sz)), T.ToTensor(), NORM])

class DS(Dataset):
    def __init__(self, ids, tf): self.ids, self.tf = ids, tf
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        j = self.ids[i]; return self.tf(imgs[j]), int(y[j])

def loader(ids, tf, train):
    return DataLoader(DS(ids, tf), batch_size=BS if train else 64, shuffle=train, drop_last=train,
                      num_workers=NUM_WORKERS, pin_memory=True)

crops 10094 | patients 2018 | train/val/test 6796/1639/1659


In [11]:
def build(arch):
    m = getattr(models, arch)(weights="DEFAULT")                   # ImageNet pretrained
    if arch == "densenet201":
        m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier.in_features, 2)); head = m.classifier
    elif arch == "inception_v3":
        m.aux_logits, m.AuxLogits = False, None
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(2048, 2)); head = m.fc
    elif arch == "resnet101":
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(2048, 2)); head = m.fc
    elif arch == "convnext_small":
        m.classifier[2] = nn.Linear(768, 2); head = m.classifier[2]
    else:                                                          # efficientnet_v2_s
        m.classifier[1] = nn.Linear(1280, 2); head = m.classifier
    return m, head

@torch.no_grad()
def predict(model, dl):                                            # P(true detection), flip TTA
    model.eval(); P = []
    for x, _ in dl:
        x = x.to(DEV, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            p = model(x).float().softmax(1) + model(x.flip(-1)).float().softmax(1)
        P.append((p[:, 1] / 2).cpu())
    return torch.cat(P).numpy()

def run(name, arch, sz, lr):
    torch.manual_seed(SEED)
    tf_tr, tf_ev = tfs(sz)
    dl_tr, dl_va, dl_te = loader(tr, tf_tr, True), loader(va, tf_ev, False), loader(te, tf_ev, False)
    model, head = build(arch); model.to(DEV)
    hid = {id(p) for p in head.parameters()}
    body = [p for p in model.parameters() if id(p) not in hid]
    opt = torch.optim.AdamW([{"params": body, "lr": lr}, {"params": list(head.parameters()), "lr": 1e-3}],
                            weight_decay=0.05)
    steps, warm = EPOCHS * len(dl_tr), len(dl_tr)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / (steps - warm))))
    scaler = torch.amp.GradScaler("cuda")
    cnt = np.bincount(y[tr], minlength=2)
    w = torch.tensor((cnt.sum() / (2 * cnt)) ** 0.5, dtype=torch.float32, device=DEV)   # mild class weights
    best, state, stale = -1, None, 0
    bar = tqdm(range(EPOCHS), desc=name, leave=False)
    for ep in bar:
        model.train()
        for x, yb in dl_tr:
            x, yb = x.to(DEV, non_blocking=True), yb.to(DEV)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = F.cross_entropy(model(x), yb, weight=w, label_smoothing=0.05)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
        auc = roc_auc_score(y[va], predict(model, dl_va))          # select on validation AUC
        bar.set_postfix(val_auc=f"{auc:.4f}")
        if auc > best: best, state, stale = auc, copy.deepcopy(model.state_dict()), 0
        else:
            stale += 1
            if stale >= PATIENCE: break
    model.load_state_dict(state); torch.save(state, OUT / f"{name}.pth")
    pv, pt = predict(model, dl_va), predict(model, dl_te)
    np.savez(OUT / f"probs_{name}.npz", pv=pv, pt=pt)
    del model, opt; torch.cuda.empty_cache()
    return pv, pt

In [12]:
def evaluate(pv, pt):
    thr = np.quantile(pv[y[va] == 1], 1 - RETENTION_TARGET)       # threshold from validation only
    yt, keep = y[te], pt >= thr
    p = np.clip(pt, 1e-7, 1 - 1e-7)
    return {"FP reduction %": (~keep[yt == 0]).mean() * 100,
            "TP retention %": keep[yt == 1].mean() * 100,
            "Accuracy %": (keep == yt).mean() * 100,
            "Loss": -np.mean(yt * np.log(p) + (1 - yt) * np.log(1 - p)),
            "AUC": roc_auc_score(yt, pt), "Threshold": thr}

res, store = {}, {}
for name, (arch, sz, lr) in MODELS.items():
    f = OUT / f"probs_{name}.npz"                                  # resumes if already trained
    if f.exists(): d = np.load(f); pv, pt = d["pv"], d["pt"]
    else: pv, pt = run(name, arch, sz, lr)
    store[name] = (pv, pt); res[name] = evaluate(pv, pt)
res["Ensemble (mean of 5)"] = evaluate(np.mean([v[0] for v in store.values()], 0),
                                       np.mean([v[1] for v in store.values()], 0))

print(f"\nTEST: {len(te)} crops, {int(y[te].sum())} true detections ({y[te].mean()*100:.1f}%) | "
      f"threshold set on validation for {RETENTION_TARGET:.0%} TP retention")
print(pd.DataFrame(res).T.round(3).to_string())

Downloading: "https://download.pytorch.org/models/densenet201-c1103571.pth" to /root/.cache/torch/hub/checkpoints/densenet201-c1103571.pth


100%|██████████| 77.4M/77.4M [00:00<00:00, 195MB/s] 


DenseNet201:   0%|          | 0/15 [00:00<?, ?it/s]

Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 229MB/s]  


InceptionV3:   0%|          | 0/15 [00:00<?, ?it/s]

... (6 outputs left out)

Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 204MB/s]


EfficientNetV2-S:   0%|          | 0/15 [00:00<?, ?it/s]


TEST: 1659 crops, 188 true detections (11.3%) | threshold set on validation for 95% TP retention
                      FP reduction %  TP retention %  Accuracy %   Loss    AUC  Threshold
DenseNet201                   19.103          89.362      27.064  0.396  0.675      0.131
InceptionV3                   11.081          98.936      21.037  0.388  0.699      0.113
ResNet101                     11.829          96.809      21.459  0.404  0.695      0.109
ConvNeXt-Small                15.976          93.085      24.714  0.464  0.694      0.135
EfficientNetV2-S              11.829          93.617      21.097  0.442  0.675      0.185
Ensemble (mean of 5)          15.432          94.149      24.352  0.410  0.709      0.155
